# TunVeCTM — Topic Modeling of Tunisian Dialect on Social Media

This notebook accompanies the research paper **"TunVeCTM: A Topic Modeling Approach of Tunisian Dialect on Social Media"** and the Master's thesis *"Topic Modeling of Tunisian Dialect on Social Media: A Contextual Approach using Combined Embeddings"* (ISG Tunis, 2025).

**Goal:** Discover coherent topics in user-generated Tunisian dialect (Arabizi / Arabic script) text from social media, using **Contextualized Topic Models (CTM)** built on top of several combined embedding strategies.

## Approach

The notebook benchmarks multiple embedding combinations feeding into a Combined Topic Model (CombinedTM):

| Section | Embedding Combination |
|---|---|
| TunBERT parameters | Baseline TunBERT contextual embeddings |
| TunBERT + FastText | Contextual + subword embeddings |
| E5 + TunBERT | Multilingual E5 sentence embeddings + TunBERT |
| SBERT + TunBERT | Multilingual SBERT + TunBERT |
| TunBERT + Doc2Vec | Contextual + document-level embeddings |
| AraBERTopic | AraBERT-based topic modeling baseline |
| Rober2vecTM | XLM-RoBERTa (Arabic fine-tuned) + Doc2Vec |

For each combination, topic coherence is evaluated across a grid of topic counts (5–100) using **C_V coherence** and **NPMI** scores to identify the best-performing configuration.

## Pipeline

1. **Data loading** — Tunisian dialect corpus (`TunTap_Corpus.xlsx`)
2. **Preprocessing** — custom Tunisian stopword removal, elongation normalization, Arabizi digit-to-letter conversion (e.g. `3` → `a`, `7` → `h`)
3. **Embedding generation** — per combination above
4. **Dimensionality reduction** — UMAP
5. **Topic modeling** — CombinedTM (Contextualized Topic Models)
6. **Evaluation** — topic coherence (C_V, NPMI) across a grid of topic counts

## Requirements

See `requirements.txt`. GPU (CUDA) is recommended for embedding generation and CTM training — this notebook was developed on Google Colab.

## Data

This notebook expects two files in the working directory (not included in this repo — see `data/README.md`):
- `TunTap_Corpus.xlsx` — the Tunisian dialect social media corpus, with a `message` column
- `tunisian_stopwords.txt` — one stopword per line

---


In [ ]:
pip install numpy==1.26.4 gensim==4.3.3 --force-reinstall --upgrade


In [ ]:
!pip install bertopic
!pip install transformers
!pip install sentence-transformers
!pip install umap-learn
!pip install hdbscan
!pip install scikit-learn
!pip install nltk
!pip install plotly
!pip install pandas
!pip install contextualized-topic-models
!pip install openpyxl

In [ ]:

import pandas as pd

# Load dataset
df = pd.read_excel("TunTap_Corpus.xlsx")

# Drop rows with missing text
df = df.dropna(subset=["message"])

# Reset index
df = df.reset_index(drop=True)

# Extract just the text column
message = df["message"].tolist()


# PRE **PROCESSING**

In [ ]:
# Read stopwords.txt into a set
with open("tunisian_stopwords.txt", "r", encoding="utf-8") as f:
    all_stopwords = set(line.strip() for line in f if line.strip())


In [ ]:
def remove_custom_stopwords(text, stopwords_set):
    tokens = text.split()
    filtered = [word for word in tokens if word not in stopwords_set]
    return " ".join(filtered)


In [ ]:
text = [remove_custom_stopwords(t, all_stopwords) for t in message]


In [ ]:
import re

def normalize_elongation(text):
    # Replace 2 or more repeated characters with 1
    return re.sub(r'(.)\1{1,}', r'\1', text)

def convert_tunisian_numbers(text): #supprimer
    return (
        text.replace("3", "a")
            .replace("7", "h")
            .replace("5", "kh")
            .replace("9", "k")
            .replace("2", "a")
    )

def remove_numbers(text):
    # Remove all digits (0-9)
    return re.sub(r'\d+', '', text)

def clean_dual_script_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove mentions and hashtags
    text = re.sub(r"@\w+|#\w+", "", text)

    # Remove emojis and symbols (non-alphanum + Arabic letters + space)
    text = re.sub(r"[^\u0621-\u063A\u0641-\u064A\w\s]", " ", text)

    # Normalize Arabic letters   #supprimer
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)


    # convert numbers like 3 → ع
    text = convert_tunisian_numbers(text)

    # Normalize elongation
    text = normalize_elongation(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()


    text = remove_numbers(text)

    return text





In [ ]:
cleaned_texts = [clean_dual_script_text(t) for t in text]

In [ ]:
final_texts = [remove_custom_stopwords(t, all_stopwords) for t in cleaned_texts]

In [ ]:
final_texts = [line for line in final_texts if line.strip() != '']


In [ ]:
print(len(final_texts))


# **TUNBERT PARAMETERS**


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np

# Load TunBERT model (classifier version)
tokenizer = AutoTokenizer.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("not-lain/TunBERT", trust_remote_code=True)
model.eval()



In [ ]:
def embed_documents(docs, batch_size=32):
    embeddings = []
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    with torch.no_grad():
        for i in range(0, len(docs), batch_size):
            batch = docs[i:i+batch_size]
            for doc in batch:
                inputs = tokenizer(doc, return_tensors="pt", truncation=True, padding=True).to(device)
                outputs = model.BertModel(**inputs, output_hidden_states=True)
                cls_embedding = outputs.last_hidden_state[:, 0, :]  # keep exactly like your code
                embeddings.append(cls_embedding.squeeze().cpu().numpy())
    return np.array(embeddings)







In [ ]:
print("Embedding documents with TunBERT...")
embeddings = embed_documents(final_texts)
print("Done embeddings. Shape:", embeddings.shape)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import torch
import umap



vectorizer_model = CountVectorizer(
    stop_words=list(all_stopwords),  # convert set to list here
    tokenizer=lambda x: x.split(),
    ngram_range=(1, 2)
)

vectorizer_model.fit(final_texts)
X_bow = vectorizer_model.transform(final_texts)  # sparse matrix

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from itertools import combinations
from collections import Counter
import math


tokenized_docs = [doc.split() for doc in message]
dictionary = Dictionary(tokenized_docs)

def compute_cv_score(topics_list, tokenized_docs, dictionary):
    cm = CoherenceModel(topics=topics_list, texts=tokenized_docs, dictionary=dictionary, coherence='c_v')
    return cm.get_coherence()

def compute_manual_npmi(topics_list, tokenized_docs):
    doc_sets = [set(d) for d in tokenized_docs]
    num_docs = len(doc_sets)
    word_doc_counts = Counter()
    for s in doc_sets:
        for w in s:
            word_doc_counts[w] += 1
    def topic_npmi(topic_words):
        vals = []
        for w1, w2 in combinations(topic_words, 2):
            p1 = word_doc_counts.get(w1, 0) / num_docs
            p2 = word_doc_counts.get(w2, 0) / num_docs
            co = sum(1 for s in doc_sets if w1 in s and w2 in s)
            p12 = co / num_docs
            if p12 > 0 and p1 > 0 and p2 > 0:
                pmi = math.log(p12 / (p1 * p2))
                npmi = pmi / (-math.log(p12))
                vals.append(npmi)
        return (sum(vals) / len(vals)) if vals else 0.0
    scores = [topic_npmi(topic) for topic in topics_list]
    return sum(scores) / len(scores), scores

# ***TUNBERT + FASTTEXT***

In [ ]:
import numpy as np
import re
import spacy
from gensim.models import FastText
from contextualized_topic_models.utils.data_preparation import CTMDataset
from contextualized_topic_models.models.ctm import CombinedTM
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from collections import Counter
from itertools import combinations
import math


# ================================
# 1. Tokenize Docs
# ================================
def simple_tokenize(text):
    return re.findall(r'\b\w+\b', text)

tokenized_docs = [simple_tokenize(doc) for doc in final_texts]

# ================================
# 2. Train FastText embeddings
# ================================
fasttext_model = FastText(
    sentences=tokenized_docs,
    vector_size=200,
    window=10,
    min_count=5,
    sg=1,          # skip-gram
    epochs=50
)
def doc_embedding(doc):
    vectors = [fasttext_model.wv[w] for w in doc if w in fasttext_model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(fasttext_model.vector_size)

fasttext_embeddings = np.array([doc_embedding(doc) for doc in tokenized_docs])

# ================================
# 3. Combine TunBERT + FastText
# ================================
# embeddings from TunBERT already exist (your contextual view)
# shape: (n_docs, d1) for tunbert, (n_docs, d2) for fasttext


# 3️⃣ Dimensionality reduction using UMAP (768 → 128)
umap_model = umap.UMAP(n_components=128, random_state=42)
tunbert_reduced = umap_model.fit_transform(embeddings)

# 4️⃣ Combine both embeddings
hybrid_embs = np.concatenate([tunbert_reduced, fasttext_embeddings], axis=1)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)

In [ ]:
# ----------------------
# 4. Build CTM dataset
# ----------------------
from contextualized_topic_models.utils.data_preparation import CTMDataset
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("not-lain/TunBERT")

training_dataset = tp.fit(
    text_for_contextual=final_texts,
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs
)

bow_size = len(tp.vocab)  # 👈 integer number of unique words
contextual_size = compressed_embs.shape[1]  # 👈 embedding dimension

# ----------------------
# 5. Train CTM
# ----------------------
from contextualized_topic_models.models.ctm import CombinedTM

ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=contextual_size,
    n_components=50,
    num_epochs=5,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3)

ctm.fit(training_dataset)

topics = ctm.get_topic_lists(10)

# ===============================
# 9. Compute Topic Coherence (CV) and NPMI
# ===============================
# Convert bag-of-words to gensim format
texts_tokenized = [doc.split() for doc in final_texts]
dictionary = Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]

topics_list = ctm.get_topic_lists(20)
cv_score = compute_cv_score(topics_list, texts_tokenized, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_list, texts_tokenized)

print("\n📊 Evaluation Metrics: of tunbert + fasttext")
print(f"CV (Topic Coherence): {cv_score:.3f}")
print(f"NPMI: {avg_npmi:.3f}")



In [ ]:
# ----------------------
# 5. Train CTM
# ----------------------
from contextualized_topic_models.models.ctm import CombinedTM

ctm = CombinedTM(
    bow_size=X_bow.shape[1],
    contextual_size=X_contextual.shape[1],
    n_components=7,
    num_epochs=10,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3)

ctm.fit(training_dataset)

topics = ctm.get_topic_lists(10)


topics_list = ctm.get_topic_lists(20)
cv_score = compute_cv_score(topics_list, texts_tokenized, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_list, texts_tokenized)

print("\n📊 Evaluation Metrics:")
print(f"CV (Topic Coherence): {cv_score:.3f}")
print(f"NPMI: {avg_npmi:.3f}")



# ***E5 + TUNBERT***

In [ ]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-base")


In [ ]:
e5_emb = e5_model.encode(final_texts, show_progress_bar=True, convert_to_numpy=True)


In [ ]:
umap_model = umap.UMAP(n_components=128, random_state=42)
e5_reduced = umap_model.fit_transform(e5_emb)
hybrid_emb = np.concatenate([e5_reduced, tunbert_reduced], axis=1)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_emb.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_emb, hybrid_emb, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs2 = encoder.predict(hybrid_emb)

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("not-lain/TunBERT")

# ✅ use fit() instead of create_dataset()
training_dataset = tp.fit(
    text_for_contextual=final_texts,  # required even if you already have embeddings
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs2     # pass your 64-d hybrid embeddings here
)


bow_size = len(tp.vocab)
contextual_size = compressed_embs2.shape[1]





In [ ]:
# ===============================
# Grid of topic numbers
# ===============================
topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]  # try any values you want
results = []

# ===============================
# Loop through topic numbers
# ===============================
for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

        # Get topics
    topics_list = ctm.get_topic_lists(20)

    # Ensure we only pass word tokens (not tuples)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    # Compute evaluation metrics
    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)

# ***SBERT + TUNBERT***

In [ ]:
sbert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


In [ ]:
sbert_emb = sbert_model.encode(final_texts, show_progress_bar=True, convert_to_numpy=True)


In [ ]:
umap_model = umap.UMAP(n_components=128, random_state=42)
sbert_reduced = umap_model.fit_transform(sbert_emb)
tunbert_reduced = umap_model.fit_transform(embeddings)
hybrid_emb2 = np.concatenate([sbert_reduced, tunbert_reduced], axis=1)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_emb2.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_emb2, hybrid_emb2, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs1 = encoder.predict(hybrid_emb2)

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("paraphrase-multilingual-MiniLM-L12-v2")

# ✅ use fit() instead of create_dataset()
training_dataset = tp.fit(
    text_for_contextual=final_texts,  # required even if you already have embeddings
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs1     # pass your 64-d hybrid embeddings here
)


bow_size = len(tp.vocab)
contextual_size = compressed_embs1.shape[1]

In [ ]:
# ===============================
# Grid of topic numbers
# ===============================
topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]  # try any values you want
results = []

# ===============================
# Loop through topic numbers
# ===============================
for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

        # Get topics
    topics_list = ctm.get_topic_lists(20)
    # Ensure we only pass word tokens (not tuples)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    # Compute evaluation metrics
    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)

# ***tunbert + doc2vec***

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument


# 1️⃣ Doc2Vec Embeddings
tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(final_texts)]
doc2vec_model = Doc2Vec(vector_size=200, window=10, min_count=5, epochs=200, sample=1e-5, workers=4)
doc2vec_model.build_vocab(tagged_data)
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
doc2vec_embs = np.array([doc2vec_model.dv[str(i)] for i in range(len(tagged_data))])


# 3️⃣ Dimensionality reduction using UMAP (768 → 128)
umap_model = umap.UMAP(n_components=128, random_state=42)

# 4️⃣ Combine both embeddings
hybrid_embs = np.concatenate([doc2vec_embs, tunbert_reduced], axis=1)

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)

In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("not-lain/TunBERT")

# ✅ use fit() instead of create_dataset()
training_dataset = tp.fit(
    text_for_contextual=final_texts,  # required even if you already have embeddings
    text_for_bow=final_texts,
    custom_embeddings=compressed_embs     # pass your 64-d hybrid embeddings here
)


bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]

In [ ]:
# ===============================
# Grid of topic numbers
# ===============================
topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]  # try any values you want
results = []

# ===============================
# Loop through topic numbers
# ===============================
for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

        # Get topics
    topics_list = ctm.get_topic_lists(20)

    # Ensure we only pass word tokens (not tuples)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    # Compute evaluation metrics
    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)

In [ ]:
    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=6,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

        # Get topics
    topics_list = ctm.get_topic_lists(20)
    for i, topic in enumerate(topics_list):
        print(f"🟢 Topic {i+1}: {', '.join(topic)}")
    # Ensure we only pass word tokens (not tuples)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    # Compute evaluation metrics
    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")




In [ ]:
import numpy as np
import pandas as pd

# 1. Get document-topic probabilities
doc_topic_probs = ctm.get_doc_topic_distribution(training_dataset)

# 2. Find dominant topic for each document
dominant_topics = np.argmax(doc_topic_probs, axis=1)

# 3. Count documents per topic
topic_counts = pd.Series(dominant_topics).value_counts().sort_index()

# 4. Get your top 20 words per topic (you already did this)
topics_list = ctm.get_topic_lists(20)

# 5. Combine into a nice table
topic_summary = pd.DataFrame({
    'Topic': range(len(topics_list)),
    'Top Words': [", ".join(words) for words in topics_list],
    'Doc Count': [topic_counts.get(t, 0) for t in range(len(topics_list))]
})

print(topic_summary)



# ***AraBERTopic***

In [ ]:
# ===============================
# 2. Imports
# ===============================
from contextualized_topic_models.models.ctm import CTM
from contextualized_topic_models.utils.data_preparation import CTMDataset
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary
import numpy as np, re, pandas as pd


# ===============================
# 3. Your cleaned texts
# ===============================
cleaned_docs = final_texts  # your preprocessed Arabic texts


# ===============================
# 4. Generate AraBERT embeddings
# ===============================
arabert = SentenceTransformer("aubmindlab/bert-base-arabertv02")  # ✅ public AraBERT v2
embeddings = arabert.encode(cleaned_docs, show_progress_bar=True, convert_to_numpy=True)


# ===============================
# 5. Create Bag-of-Words
# ===============================
vectorizer = CountVectorizer(max_features=2000)
bow = vectorizer.fit_transform(cleaned_docs)
id2word = {i: w for i, w in enumerate(vectorizer.get_feature_names_out())}


In [ ]:
import umap
umap_model = umap.UMAP(n_components=128, random_state=42)
arabert_reduced = umap_model.fit_transform(embeddings)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = arabert_reduced.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(arabert_reduced, arabert_reduced, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(arabert_reduced)

In [ ]:
# ===============================
# 6. Prepare dataset (manually compatible)
# ===============================
# Newer versions of CTMDataset expect only bow + vocab, not contextualized embeddings
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

# We store embeddings separately and feed them directly during CTM training
tp = TopicModelDataPreparation("aubmindlab/bert-base-arabertv02")

# ✅ use fit() instead of create_dataset()
training_dataset = tp.fit(
    text_for_contextual=cleaned_docs,  # required even if you already have embeddings
    text_for_bow=cleaned_docs,
    custom_embeddings=compressed_embs     # pass your 64-d hybrid embeddings here
)


bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]  # should be 64



In [ ]:
# ===============================
# Grid of topic numbers
# ===============================
from contextualized_topic_models.models.ctm import CombinedTM


topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]  # try any values you want
results = []

# ===============================
# Loop through topic numbers
# ===============================
for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    prodlda_arabert = CombinedTM(
     bow_size=bow_size,
     contextual_size=contextual_size,
     n_components=n_topics,
     num_epochs=5,
     batch_size=64,# number of topics
 )

    prodlda_arabert.fit(training_dataset)

    topics = prodlda_arabert.get_topic_lists(20)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics
     ]


    texts_tokenized = [doc.split() for doc in cleaned_docs]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)

# ***Rober2vecTM***

In [ ]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from transformers import AutoTokenizer, AutoModel
import torch
import umap
import numpy as np

processed_texts = final_texts

# 1️⃣ Doc2Vec Embeddings
tagged_data = [TaggedDocument(words=_d.split(), tags=[str(i)]) for i, _d in enumerate(processed_texts)]
doc2vec_model = Doc2Vec(vector_size=200, window=10, min_count=5, epochs=50, sample=1e-5, workers=4)
doc2vec_model.build_vocab(tagged_data)
doc2vec_model.train(tagged_data, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)
doc2vec_embs = np.array([doc2vec_model.dv[str(i)] for i in range(len(tagged_data))])

# 2️⃣ RoBERTa Embeddings (fine-tuned on Arabic)
tokenizer = AutoTokenizer.from_pretrained("Davlan/xlm-roberta-base-finetuned-arabic")
model = AutoModel.from_pretrained("Davlan/xlm-roberta-base-finetuned-arabic")

def roberta_embed(texts):
    embs = []
    for t in texts:
        inputs = tokenizer(t, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embs.append(emb)
    return np.array(embs)

roberta_embs = roberta_embed(processed_texts)

# 3️⃣ Dimensionality reduction using UMAP (768 → 128)
umap_model = umap.UMAP(n_components=128, random_state=42)
roberta_reduced = umap_model.fit_transform(roberta_embs)

# 4️⃣ Combine both embeddings
hybrid_embs = np.concatenate([doc2vec_embs, roberta_reduced], axis=1)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

input_dim = hybrid_embs.shape[1]
input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation='relu')(input_layer)
decoded = Dense(input_dim, activation='linear')(encoded)
autoencoder = Model(input_layer, decoded)

autoencoder.compile(optimizer=Adam(), loss='mse')
autoencoder.fit(hybrid_embs, hybrid_embs, epochs=200, batch_size=32, verbose=1)

encoder = Model(input_layer, encoded)
compressed_embs = encoder.predict(hybrid_embs)


In [ ]:
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation

tp = TopicModelDataPreparation("Davlan/xlm-roberta-base-finetuned-arabic")

# ✅ use fit() instead of create_dataset()
training_dataset = tp.fit(
    text_for_contextual=processed_texts,  # required even if you already have embeddings
    text_for_bow=processed_texts,
    custom_embeddings=compressed_embs     # pass your 64-d hybrid embeddings here
)


bow_size = len(tp.vocab)
contextual_size = compressed_embs.shape[1]  # should be 64



contextual_size = compressed_embs.shape[1]

ctm = CombinedTM(
    bow_size=bow_size,
    contextual_size=contextual_size,
    n_components=7,
    num_epochs=5,
    batch_size=64,
    hidden_sizes=(128,),
    activation="relu",
    dropout=0.0,
    lr=2e-3
)


ctm.fit(training_dataset)


In [ ]:
topics = ctm.get_topic_lists(10)
for i, topic in enumerate(topics):
    print(f"🟢 Topic {i+1}: {', '.join(topic)}")

# ===============================
# 9. Compute Topic Coherence (CV) and NPMI
# ===============================
# Convert bag-of-words to gensim format
texts_tokenized = [doc.split() for doc in processed_texts]
dictionary = Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]

topics_list = ctm.get_topic_lists(20)
cv_score = compute_cv_score(topics_list, texts_tokenized, dictionary)
avg_npmi, _ = compute_manual_npmi(topics_list, texts_tokenized)

print("\n📊 Evaluation Metrics:")
print(f"CV (Topic Coherence): {cv_score:.3f}")
print(f"NPMI: {avg_npmi:.3f}")

In [ ]:
# ===============================
# Grid of topic numbers
# ===============================
from contextualized_topic_models.models.ctm import CombinedTM


topic_numbers = [5, 6, 7, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 95, 100]  # try any values you want
results = []

# ===============================
# Loop through topic numbers
# ===============================
for n_topics in topic_numbers:
    print(f"\n🌀 Training CombinedTM with {n_topics} topics...")

    ctm = CombinedTM(
        bow_size=bow_size,
        contextual_size=contextual_size,
        n_components=n_topics,
        num_epochs=5,
        batch_size=64,
        hidden_sizes=(128,),
        activation="relu",
        dropout=0.0,
        lr=2e-3
    )

    ctm.fit(training_dataset)

        # Get topics
    topics_list = ctm.get_topic_lists(20)

    # Ensure we only pass word tokens (not tuples)
    topics_clean = [
        [w if isinstance(w, str) else w[0] for w in topic]
        for topic in topics_list
    ]

    # Compute evaluation metrics
    texts_tokenized = [doc.split() for doc in final_texts]
    dictionary = Dictionary(texts_tokenized)
    corpus = [dictionary.doc2bow(text) for text in texts_tokenized]
    cv_score = compute_cv_score(topics_clean, texts_tokenized, dictionary)
    avg_npmi, _ = compute_manual_npmi(topics_clean, texts_tokenized)

    print(f"✅ {n_topics} topics → CV: {cv_score:.3f}, NPMI: {avg_npmi:.3f}")

    results.append((n_topics, cv_score, avg_npmi))

# ===============================
# Display results as table
# ===============================
results_df = pd.DataFrame(results, columns=["n_topics", "CV", "NPMI"])
print("\n📊 Grid Search Results:")
print(results_df)